# Process External Volume Interface Transcript

Process the pending transcript end-to-end:
1. Read and clean the raw transcript
2. Split into topic blocks
3. Match against existing Knowledge Objects
4. Create/update KOs as needed
5. Save extraction artifact

In [ ]:
import sys
import os
import json
import re
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple

# Add pipeline to path
sys.path.insert(0, r'c:\Hackathon\AgentVerse')

from pipeline.extract import extract
from pipeline.clean import clean

# Configuration
TRANSCRIPT_FILE = r'c:\Hackathon\AgentVerse\transcripts\pending\video_66fo5e3lkqt_19_160_1220x686_transcript.txt'
KO_DIR = Path(r'c:\Hackathon\AgentVerse\knowledge\objects')
ARTIFACT_DIR = Path(r'c:\Hackathon\AgentVerse\knowledge\artifacts\run-20260922-external-volume-interface')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Configuration loaded")

## 1. Load and Clean Raw Transcript

In [ ]:
# Step 1: Extract and clean the transcript
raw_text = extract(TRANSCRIPT_FILE)
raw_char_count = len(raw_text)

# Clean the transcript
clean_text = clean(raw_text)
clean_char_count = len(clean_text)

# Record source metadata
source_metadata = {
    "source_file": Path(TRANSCRIPT_FILE).name,
    "format": "txt",
    "extracted_at": datetime.now().isoformat(),
    "raw_char_count": raw_char_count,
    "clean_char_count": clean_char_count,
    "clean_text": clean_text
}

print(f"Raw text: {raw_char_count} characters")
print(f"Clean text: {clean_char_count} characters")
print(f"Reduction: {(1 - clean_char_count/raw_char_count)*100:.1f}%")
print(f"\nFirst 500 characters of clean text:\n{clean_text[:500]}")

## 2. Split Transcript into Topic Blocks

In [ ]:
# Split into paragraphs (potential topic blocks)
paragraphs = clean_text.split('\n\n')

# Filter out short fragments (< 20 words) and empty paragraphs
def word_count(text):
    return len(text.split())

topic_blocks = []
for para in paragraphs:
    if para.strip() and word_count(para) >= 20:
        topic_blocks.append(para.strip())

print(f"Total paragraphs: {len(paragraphs)}")
print(f"Topic blocks (>= 20 words): {len(topic_blocks)}")
print(f"\nTopic blocks breakdown:")
for i, block in enumerate(topic_blocks, 1):
    wc = word_count(block)
    first_sentence = block.split('.')[0] if '.' in block else block[:60]
    print(f"{i}. [{wc} words] {first_sentence}...")

## 3. Generate Section Headings and Key Points

In [ ]:
def extract_heading(block: str, max_chars: int = 60) -> str:
    """Extract a heading from the first sentence of a block."""
    # Get first sentence (ends with . ! or ?)
    match = re.match(r'^([^.!?]*[.!?])', block)
    if match:
        heading = match.group(1).strip()
        # Remove trailing punctuation and truncate
        heading = heading.rstrip('.!?').strip()
        if len(heading) > max_chars:
            heading = heading[:max_chars].rsplit(' ', 1)[0]
        return heading
    # Fallback: use first 60 chars
    return block[:max_chars].strip()

def extract_key_points(block: str, max_points: int = 5) -> List[str]:
    """Extract key points from a block."""
    # Split by periods or natural breaks
    sentences = re.split(r'[.!?]\s+', block)
    key_points = []
    
    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) > 20 and len(key_points) < max_points:
            # Clean up the sentence
            sentence = re.sub(r'^[^a-z]*', '', sentence, flags=re.IGNORECASE)
            if sentence:
                key_points.append(sentence)
    
    return key_points

# Process each topic block
topics = []
for i, block in enumerate(topic_blocks, 1):
    heading = extract_heading(block)
    key_points = extract_key_points(block)
    
    topic = {
        "index": i,
        "heading": heading,
        "key_points": key_points,
        "body": block[:200] + "..." if len(block) > 200 else block
    }
    topics.append(topic)

print(f"Extracted {len(topics)} topics with headings and key points:\n")
for topic in topics:
    print(f"{topic['index']}. {topic['heading']}")
    for kp in topic['key_points'][:2]:
        print(f"   - {kp[:70]}...")
    print()

## 4. Load Existing Knowledge Objects

In [ ]:
# Load all existing Knowledge Objects
existing_kos = {}
ko_files = sorted(KO_DIR.glob('ko-*.json'))

for ko_file in ko_files:
    try:
        with open(ko_file, 'r') as f:
            ko = json.load(f)
            existing_kos[ko['id']] = ko
    except Exception as e:
        print(f"Error loading {ko_file}: {e}")

print(f"Loaded {len(existing_kos)} existing Knowledge Objects")
print(f"\nFirst 10 KOs:")
for ko_id in list(existing_kos.keys())[:10]:
    title = existing_kos[ko_id].get('title', 'N/A')
    category = existing_kos[ko_id].get('category', 'N/A')
    print(f"  - {ko_id}: {title} ({category})")

## 5. Match Topics Against Knowledge Objects

In [ ]:
def simple_similarity(text1: str, text2: str) -> float:
    """Calculate simple text similarity based on shared terms."""
    words1 = set(text1.lower().split())
    words2 = set(text2.lower().split())
    
    if not words1 or not words2:
        return 0.0
    
    intersection = len(words1 & words2)
    union = len(words1 | words2)
    
    return intersection / union if union > 0 else 0.0

# Match each topic against existing KOs
topic_matches = []
for topic in topics:
    topic_text = topic['heading'] + ' ' + ' '.join(topic['key_points'])
    
    best_match = None
    best_score = 0.0
    
    for ko_id, ko in existing_kos.items():
        # Build search text from KO title, tags, and section headings
        ko_text = ko['title'] + ' ' + ' '.join(ko.get('tags', []))
        for section in ko.get('sections', []):
            ko_text += ' ' + section['heading']
        
        score = simple_similarity(topic_text, ko_text)
        if score > best_score:
            best_score = score
            best_match = (ko_id, ko, score)
    
    # Determine match type based on score
    if best_score >= 0.45:  # Similarity threshold for existing-ko
        match_type = "existing-ko"
        matched_ko_id = best_match[0]
    else:
        # Determine if we have a matching category
        match_type = "new-ko-existing-category"
        matched_ko_id = None
        
        # Check for category matches
        topic_lower = topic_text.lower()
        for ko_id, ko in existing_kos.items():
            category = ko.get('category', '')
            if any(word in category for word in topic['heading'].lower().split()):
                match_type = "new-ko-existing-category"
                break
    
    topic_matches.append({
        "topic_index": topic['index'],
        "heading": topic['heading'],
        "match_type": match_type,
        "matched_ko_id": matched_ko_id,
        "match_score": best_score if best_match else 0.0
    })

print("Topic Matching Results:\n")
for match in topic_matches:
    print(f"{match['topic_index']}. {match['heading'][:50]}")
    print(f"   Match type: {match['match_type']}")
    print(f"   Matched KO: {match['matched_ko_id']}")
    print(f"   Score: {match['match_score']:.3f}\n")

## 6. Create or Update Knowledge Objects

In [ ]:
def slugify(title: str) -> str:
    """Convert title to kebab-case slug."""
    slug = title.lower()
    slug = re.sub(r'[^a-z0-9\s-]', '', slug)
    slug = re.sub(r'\s+', '-', slug).strip('-')
    slug = re.sub(r'-+', '-', slug)
    return slug[:50]  # Limit slug length

def create_ko_section(topic: dict) -> dict:
    """Create a knowledge object section from a topic."""
    return {
        "heading": topic['heading'],
        "body": topic_blocks[topic['index'] - 1],  # Full block as body
        "key_points": topic['key_points']
    }

# Track created and updated KOs
created_kos = []
updated_kos = []

for match in topic_matches:
    topic = topics[match['topic_index'] - 1]
    
    if match['match_type'] == 'existing-ko':
        # Update existing KO
        ko_id = match['matched_ko_id']
        ko = existing_kos[ko_id]
        
        # Check if section already exists (avoid duplicates)
        section_exists = any(
            simple_similarity(section['heading'], topic['heading']) > 0.75
            for section in ko['sections']
        )
        
        if not section_exists:
            # Append new section
            ko['sections'].append(create_ko_section(topic))
            ko['version'] = ko.get('version', 1) + 1
            ko['updated_at'] = datetime.now().strftime('%Y-%m-%d')
            
            # Append to sources if not already there
            new_source = {
                "transcript": Path(TRANSCRIPT_FILE).name,
                "extracted_at": datetime.now().strftime('%Y-%m-%d')
            }
            
            source_exists = any(
                s['transcript'] == new_source['transcript']
                for s in ko.get('sources', [])
            )
            if not source_exists:
                ko['sources'] = ko.get('sources', []) + [new_source]
            
            updated_kos.append(ko_id)
            
            # Save updated KO
            ko_file = KO_DIR / f"{ko_id}.json"
            with open(ko_file, 'w') as f:
                json.dump(ko, f, indent=2)
    
    elif match['match_type'] == 'new-ko-existing-category':
        # Create new KO
        slug = slugify(topic['heading'])
        ko_id = f"ko-{slug}"
        
        # Avoid ID conflicts
        counter = 1
        while ko_id in existing_kos or (KO_DIR / f"{ko_id}.json").exists():
            ko_id = f"ko-{slug}-{counter}"
            counter += 1
        
        # Determine category - use imports as default
        category = "core-features/imports.md"
        
        new_ko = {
            "id": ko_id,
            "slug": slug,
            "title": topic['heading'],
            "category": category,
            "status": "draft",
            "summary": topic['key_points'][0] if topic['key_points'] else topic['heading'],
            "sections": [create_ko_section(topic)],
            "tags": ["external-volume", "manual-entry", "meter-input"],
            "sources": [{
                "transcript": Path(TRANSCRIPT_FILE).name,
                "extracted_at": datetime.now().strftime('%Y-%m-%d')
            }],
            "relationships": [],
            "faqs": [],
            "version": 1,
            "created_at": datetime.now().strftime('%Y-%m-%d'),
            "updated_at": datetime.now().strftime('%Y-%m-%d'),
            "published_pages": []
        }
        
        created_kos.append(ko_id)
        existing_kos[ko_id] = new_ko
        
        # Save new KO
        ko_file = KO_DIR / f"{ko_id}.json"
        with open(ko_file, 'w') as f:
            json.dump(new_ko, f, indent=2)

print(f"Created {len(created_kos)} new Knowledge Objects:")
for ko_id in created_kos:
    print(f"  - {ko_id}")

print(f"\nUpdated {len(updated_kos)} Knowledge Objects:")
for ko_id in updated_kos:
    print(f"  - {ko_id}")

## 7. Save Extraction Artifact

In [ ]:
# Build extraction artifact
extraction_artifact = {
    "run_id": "run-20260922-external-volume-interface",
    "agent": "knowledge-extraction",
    "generated_at": datetime.now().isoformat(),
    "source_transcript": Path(TRANSCRIPT_FILE).name,
    "topics": topic_matches,
    "knowledge_objects": created_kos + updated_kos,
    "unmatched_topics": []  # All topics were matched
}

# Save artifact
artifact_file = ARTIFACT_DIR / "artifact-extraction.schema.json"
with open(artifact_file, 'w') as f:
    json.dump(extraction_artifact, f, indent=2)

print(f"✓ Extraction artifact saved to: {artifact_file}")
print(f"\nArtifact summary:")
print(f"  - Run ID: {extraction_artifact['run_id']}")
print(f"  - Generated: {extraction_artifact['generated_at']}")
print(f"  - Topics extracted: {len(extraction_artifact['topics'])}")
print(f"  - Knowledge Objects created: {len(created_kos)}")
print(f"  - Knowledge Objects updated: {len(updated_kos)}")

## 8. Generate Processing Summary

In [ ]:
print("=" * 70)
print("KNOWLEDGE EXTRACTION PROCESSING SUMMARY")
print("=" * 70)
print(f"\nTranscript: {Path(TRANSCRIPT_FILE).name}")
print(f"Processing Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\n--- Transcript Statistics ---")
print(f"Raw text: {raw_char_count:,} characters")
print(f"Clean text: {clean_char_count:,} characters")
print(f"Reduction: {(1 - clean_char_count/raw_char_count)*100:.1f}%")

print(f"\n--- Topic Extraction ---")
print(f"Total topics extracted: {len(topics)}")
print(f"Topics matched to existing KOs: {sum(1 for m in topic_matches if m['match_type'] == 'existing-ko')}")
print(f"Topics needing new KOs: {sum(1 for m in topic_matches if m['match_type'] == 'new-ko-existing-category')}")
print(f"Topics with new categories: {sum(1 for m in topic_matches if m['match_type'] == 'new-category')}")

print(f"\n--- Knowledge Objects ---")
print(f"Total existing KOs: {len(existing_kos)}")
print(f"KOs created (new): {len(created_kos)}")
print(f"KOs updated: {len(updated_kos)}")
print(f"KOs affected: {len(created_kos) + len(updated_kos)}")

print(f"\n--- Created Knowledge Objects ---")
if created_kos:
    for ko_id in created_kos:
        print(f"  • {ko_id}")
else:
    print("  (None)")

print(f"\n--- Updated Knowledge Objects ---")
if updated_kos:
    for ko_id in updated_kos:
        print(f"  • {ko_id}")
else:
    print("  (None)")

print(f"\n--- Artifact ---")
print(f"Saved to: {artifact_file}")
print(f"Run ID: run-20260922-external-volume-interface")

print("\n" + "=" * 70)
print("EXTRACTION COMPLETE")
print("=" * 70)